In [ ]:
# ==========================================
# STEP 0: Auto-Install Dependencies & Silence Warnings
# ==========================================
import os
import gc
import warnings
import logging
import math
import numpy as np
import scipy.stats as stats

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

print("Installing dependencies... (This takes about 30 seconds)")
os.system("pip install -q beir sentence-transformers transformers torch numpy scipy tqdm")

import huggingface_hub
import transformers
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F_func
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModel
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from tqdm import tqdm

huggingface_hub.utils.logging.set_verbosity_error()
transformers.logging.set_verbosity_error()
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n--- System is utilizing: {DEVICE.type.upper()} ---")

# ==========================================
# ISR Architecture & Training
# ==========================================
class ISR_Bottleneck(nn.Module):
    def __init__(self, dense_dim=768, sparse_dim=4096):
        super(ISR_Bottleneck, self).__init__()
        self.encoder = nn.Linear(dense_dim, sparse_dim)
        self.relu = nn.ReLU()
        self.decoder = nn.Linear(sparse_dim, dense_dim)

    def forward(self, dense_embeds):
        F = self.relu(self.encoder(dense_embeds))
        reconstructed_E = self.decoder(F)
        return F, reconstructed_E

def train_isr_bottleneck(isr_model, train_embeddings, epochs=150, l1_lambda=1.5e-6, ip_lambda=3.0):
    optimizer = optim.Adam(isr_model.parameters(), lr=1e-3)
    mse_loss_fn = nn.MSELoss()
    isr_model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        F, reconstructed_E = isr_model(train_embeddings)
        reconstruction_loss = mse_loss_fn(reconstructed_E, train_embeddings)
        sparsity_penalty = l1_lambda * torch.norm(F, p=1)

        F_norm = F_func.normalize(F, p=2, dim=1, eps=1e-8)
        sim_F = torch.matmul(F_norm, F_norm.T)
        sim_E = torch.matmul(train_embeddings, train_embeddings.T)
        ip_loss = mse_loss_fn(sim_F, sim_E)

        total_loss = reconstruction_loss + sparsity_penalty + (ip_lambda * ip_loss)
        total_loss.backward()
        optimizer.step()
    return isr_model

# ==========================================
# Metric Utils
# ==========================================
def compute_ndcg_at_k(retrieved_indices, relevant_indices, k=10):
    dcg = 0.0
    for i, idx in enumerate(retrieved_indices[:k]):
        if idx in relevant_indices:
            dcg += 1.0 / math.log2(i + 2)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant_indices), k)))
    return dcg / idcg if idcg > 0 else 0.0

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ==========================================
# Evaluation Modules
# ==========================================
def evaluate_splade(sample_docs, sample_queries, query_ids, qrels, doc_id_to_idx):
    print("  -> Loading SPLADE...")
    tokenizer = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
    model = AutoModelForMaskedLM.from_pretrained("naver/splade-cocondenser-ensembledistil").to(DEVICE)
    model.eval()

    def get_splade_vectors(texts):
        tokens = tokenizer(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            output = model(**tokens)
        vecs = torch.max(torch.log(1 + torch.relu(output.logits)) * tokens.attention_mask.unsqueeze(-1), dim=1)[0]
        return vecs

    print("  -> Encoding SPLADE vectors...")
    d_vecs = []
    batch_size = 16
    for i in tqdm(range(0, len(sample_docs), batch_size), desc="SPLADE Docs"):
        d_vecs.append(get_splade_vectors(sample_docs[i:i+batch_size]))
    d_vecs = torch.cat(d_vecs, dim=0)
    q_vecs = get_splade_vectors(sample_queries)
    scores = torch.matmul(q_vecs, d_vecs.T)

    ndcg_list = []
    dts_list = []
    for i, qid in enumerate(query_ids):
        if qid not in qrels: continue
        relevant_idx = [doc_id_to_idx[did] for did in qrels[qid].keys() if did in doc_id_to_idx]
        if not relevant_idx: continue

        top_k_idx = torch.topk(scores[i], k=10).indices.tolist()
        ndcg_list.append(compute_ndcg_at_k(top_k_idx, relevant_idx, k=10))

        query_primary_token = torch.argmax(q_vecs[i]).item()
        dts_matches = sum(1 for idx in top_k_idx if query_primary_token in torch.topk(d_vecs[idx], k=3).indices.tolist())
        dts_list.append((dts_matches / 10.0) * 100)

    active_dims = (d_vecs > 0).float().sum(dim=1).mean().item()
    sparsity_pct = (1.0 - (active_dims / model.config.vocab_size)) * 100

    del model, tokenizer, d_vecs, q_vecs, scores
    clear_vram()
    return ndcg_list, np.mean(dts_list), sparsity_pct

def evaluate_colbert(sample_docs, sample_queries, query_ids, qrels, doc_id_to_idx):
    print("  -> Loading ColBERTv2.0...")
    tokenizer = AutoTokenizer.from_pretrained("colbert-ir/colbertv2.0")
    model = AutoModel.from_pretrained("colbert-ir/colbertv2.0").to(DEVICE)
    model.eval()

    def get_token_embeddings(texts):
        tokens = tokenizer(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(DEVICE)
        with torch.no_grad():
            outputs = model(**tokens).last_hidden_state
        return F_func.normalize(outputs, p=2, dim=2), tokens.attention_mask

    print("  -> Encoding ColBERT tokens...")
    q_embs, q_masks = get_token_embeddings(sample_queries)

    d_embs_list, d_masks_list = [], []
    batch_size = 16
    for i in tqdm(range(0, len(sample_docs), batch_size), desc="ColBERT Docs"):
        embs, masks = get_token_embeddings(sample_docs[i:i+batch_size])
        d_embs_list.append(embs)
        d_masks_list.append(masks)
    d_embs = torch.cat(d_embs_list, dim=0)
    d_masks = torch.cat(d_masks_list, dim=0)

    d_pooled = d_embs.max(dim=1).values
    q_pooled = q_embs.max(dim=1).values
    d_centered = d_pooled - d_pooled.mean(dim=0)
    q_centered = q_pooled - d_pooled.mean(dim=0)

    ndcg_list = []
    dts_list = []
    print("  -> Computing MaxSim...")
    for i, qid in enumerate(query_ids):
        if qid not in qrels: continue
        relevant_idx = [doc_id_to_idx[did] for did in qrels[qid].keys() if did in doc_id_to_idx]
        if not relevant_idx: continue

        q_i = q_embs[i].unsqueeze(0)
        sim_matrix = torch.einsum("b q d, c d -> b q c", q_i, d_embs.view(-1, 768))
        sim_matrix = sim_matrix.view(1, q_embs.shape[1], d_embs.shape[0], d_embs.shape[1])

        d_mask_expanded = d_masks.unsqueeze(0).unsqueeze(1)
        sim_matrix = sim_matrix.masked_fill(d_mask_expanded == 0, -1e4)
        max_sim_per_q_token = sim_matrix.max(dim=3).values

        q_mask_expanded = q_masks[i].unsqueeze(0).unsqueeze(-1)
        doc_scores = (max_sim_per_q_token * q_mask_expanded).sum(dim=1).squeeze(0)

        top_k_idx = torch.topk(doc_scores, k=10).indices.tolist()
        ndcg_list.append(compute_ndcg_at_k(top_k_idx, relevant_idx, k=10))

        query_primary_dense_feature = torch.argmax(q_centered[i]).item()
        colbert_dts_matches = sum(1 for idx in top_k_idx if torch.argmax(d_centered[idx]).item() == query_primary_dense_feature)
        dts_list.append((colbert_dts_matches / 10.0) * 100)

    del model, tokenizer, q_embs, d_embs, sim_matrix
    clear_vram()
    return ndcg_list, np.mean(dts_list)

# ==========================================
# Main Execution
# ==========================================
def run_experiment():
    SEED = 42
    import random
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    # ADDED FiQA and TREC-COVID
    target_datasets = ["scidocs", "arguana", "nfcorpus", "fiqa", "trec-covid"]
    results = {}

    for dataset in target_datasets:
        print(f"\n==================================================")
        print(f" PROCESSING DATASET: {dataset.upper()}")
        print(f"==================================================")

        url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"
        out_dir = util.download_and_unzip(url, "datasets")

        corpus, queries, qrels = GenericDataLoader(data_folder=out_dir).load(split="test")

        query_ids = list(queries.keys())[:20]
        relevant_doc_ids = []
        for qid in query_ids:
            if qid in qrels:
                relevant_doc_ids.extend([did for did, score in qrels[qid].items() if score > 0])

        all_doc_ids = list(set(relevant_doc_ids + list(corpus.keys())[:200]))
        doc_id_to_idx = {did: i for i, did in enumerate(all_doc_ids)}

        sample_docs = [corpus[did].get("text") for did in all_doc_ids]
        sample_queries = [queries[qid] for qid in query_ids]

        # 1. EVALUATE SPLADE
        splade_ndcgs, splade_dts, splade_sparsity = evaluate_splade(sample_docs, sample_queries, query_ids, qrels, doc_id_to_idx)

        # 2. EVALUATE COLBERT
        colbert_ndcgs, colbert_dts = evaluate_colbert(sample_docs, sample_queries, query_ids, qrels, doc_id_to_idx)

        # 3. EVALUATE CONTRIEVER & ISR
        print("  -> Loading Contriever Base...")
        base_model = SentenceTransformer('facebook/contriever', device=DEVICE)

        d_dense = torch.tensor(base_model.encode(sample_docs, batch_size=64, show_progress_bar=True, convert_to_tensor=True)).to(DEVICE)
        d_dense = F_func.normalize(d_dense, p=2, dim=1)

        q_dense = torch.tensor(base_model.encode(sample_queries, batch_size=64, show_progress_bar=True, convert_to_tensor=True)).to(DEVICE)
        q_dense = F_func.normalize(q_dense, p=2, dim=1)

        baseline_scores = torch.matmul(q_dense, d_dense.T)
        baseline_ndcg, baseline_dts_scores = [], []

        d_dense_centered = d_dense - d_dense.mean(dim=0)
        q_dense_centered = q_dense - d_dense.mean(dim=0)

        for i, qid in enumerate(query_ids):
            if qid not in qrels: continue
            relevant_idx = [doc_id_to_idx[did] for did in qrels[qid].keys() if did in doc_id_to_idx]
            if not relevant_idx: continue

            top_k_idx = torch.topk(baseline_scores[i], k=10).indices.tolist()
            baseline_ndcg.append(compute_ndcg_at_k(top_k_idx, relevant_idx, k=10))

            query_primary_dense_feature = torch.argmax(q_dense_centered[i]).item()
            base_dts_matches = sum(1 for idx in top_k_idx if torch.argmax(d_dense_centered[idx]).item() == query_primary_dense_feature)
            baseline_dts_scores.append((base_dts_matches / 10.0) * 100)

        print("  -> Training Interpretable Semantic Resonance (ISR)...")
        train_embeddings = torch.cat([d_dense, q_dense], dim=0)
        isr_layer = ISR_Bottleneck(dense_dim=768, sparse_dim=4096).to(DEVICE)
        isr_layer = train_isr_bottleneck(isr_layer, train_embeddings, epochs=150, l1_lambda=1.5e-6, ip_lambda=3.0)
        isr_layer.eval()

        with torch.no_grad():
            q_F, _ = isr_layer(q_dense)
            d_F, _ = isr_layer(d_dense)

        q_F_norm = F_func.normalize(q_F, p=2, dim=1, eps=1e-8)
        d_F_norm = F_func.normalize(d_F, p=2, dim=1, eps=1e-8)

        isr_scores = torch.matmul(q_F_norm, d_F_norm.T)
        isr_ndcg, isr_dts_scores, mci_steps_list = [], [], []

        for i, qid in enumerate(query_ids):
            if qid not in qrels: continue
            relevant_idx = [doc_id_to_idx[did] for did in qrels[qid].keys() if did in doc_id_to_idx]
            if not relevant_idx: continue

            top_k_idx = torch.topk(isr_scores[i], k=10).indices.tolist()
            isr_ndcg.append(compute_ndcg_at_k(top_k_idx, relevant_idx, k=10))

            query_primary_facet = torch.argmax(q_F_norm[i]).item()
            dts_matches = sum(1 for idx in top_k_idx if query_primary_facet in torch.topk(d_F_norm[idx], k=3).indices.tolist())
            isr_dts_scores.append((dts_matches / 10.0) * 100)

            target_doc_idx = relevant_idx[0]
            initial_rank = (torch.argsort(isr_scores[i], descending=True) == target_doc_idx).nonzero(as_tuple=True)[0].item() + 1

            if initial_rank > 10:
                steps = 0
                M = torch.ones_like(q_F_norm[i])
                target_dominant_facet = torch.argmax(d_F_norm[target_doc_idx]).item()
                while steps < 10:
                    steps += 1
                    M[target_dominant_facet] += 2.0
                    adjusted_q = q_F_norm[i] * M
                    new_scores = torch.matmul(adjusted_q.unsqueeze(0), d_F_norm.T).squeeze(0)
                    new_rank = (torch.argsort(new_scores, descending=True) == target_doc_idx).nonzero(as_tuple=True)[0].item() + 1
                    if new_rank <= 10: break
                mci_steps_list.append(steps)

        def get_p_value(base_scores, comparison_scores):
            if len(base_scores) > 1 and len(comparison_scores) > 1:
                return stats.ttest_rel(base_scores, comparison_scores)[1]
            return 1.0

        pval_colbert = get_p_value(baseline_ndcg, colbert_ndcgs)
        pval_splade = get_p_value(baseline_ndcg, splade_ndcgs)
        pval_isr = get_p_value(baseline_ndcg, isr_ndcg)

        active_dims_per_doc = (d_F_norm > 0).float().sum(dim=1).mean().item()
        isr_sparsity_pct = (1.0 - (active_dims_per_doc / 4096.0)) * 100

        results[dataset] = {
            "base_ndcg": np.mean(baseline_ndcg),
            "base_dts": np.mean(baseline_dts_scores),

            "colbert_ndcg": np.mean(colbert_ndcgs),
            "colbert_dts": colbert_dts,
            "colbert_pval": pval_colbert,

            "splade_ndcg": np.mean(splade_ndcgs),
            "splade_dts": splade_dts,
            "splade_sparsity": splade_sparsity,
            "splade_pval": pval_splade,

            "isr_ndcg": np.mean(isr_ndcg),
            "isr_dts": np.mean(isr_dts_scores),
            "isr_mci": np.mean(mci_steps_list) if mci_steps_list else 1.0,
            "isr_pval": pval_isr,
            "isr_sparsity": isr_sparsity_pct
        }

        del base_model, d_dense, q_dense, isr_layer
        clear_vram()

    # --- FINAL TABLE OUTPUT ---
    print("\n\n=======================================================================================================================")
    print(" Table 1: Retrieval, Controllability, and Hardware Footprint Across BEIR Subsets")
    print("=======================================================================================================================")
    print(f"{'Dataset':<10} | {'Model':<23} | {'NDCG@10':<9} | {'DTS (%)':<9} | {'MCI':<7} | {'p-value':<10} | {'Zero-Vals (%)'}")
    print("-" * 119)

    for ds in target_datasets:
        data_name = ds.capitalize()

        def format_pval(pval):
            return f"{pval:<6.3f} ({'*' if pval < 0.05 else 'ns'})"

        print(f"{data_name:<10} | {'Contriever (Dense)':<23} | {results[ds]['base_ndcg']:<9.4f} | {results[ds]['base_dts']:<9.1f} | {'N/A':<7} | {'-':<10} | {'0.0% (Dense)'}")
        print(f"{'':<10} | {'ColBERTv2.0 (Dense)':<23} | {results[ds]['colbert_ndcg']:<9.4f} | {results[ds]['colbert_dts']:<9.1f} | {'N/A':<7} | {format_pval(results[ds]['colbert_pval']):<10} | {'0.0% (Dense)'}")
        print(f"{'':<10} | {'SPLADE (Sparse)':<23} | {results[ds]['splade_ndcg']:<9.4f} | {results[ds]['splade_dts']:<9.1f} | {'N/A':<7} | {format_pval(results[ds]['splade_pval']):<10} | {results[ds]['splade_sparsity']:.1f}%")
        print(f"{'':<10} | {'ISR-Contriever (Ours)':<23} | {results[ds]['isr_ndcg']:<9.4f} | {results[ds]['isr_dts']:<9.1f} | {results[ds]['isr_mci']:<7.1f} | {format_pval(results[ds]['isr_pval']):<10} | {results[ds]['isr_sparsity']:.1f}%")
        print("-" * 119)
    print("=======================================================================================================================")
    print("* Note: 'ns' indicates Not Significant (p >= 0.05) vs Contriever Baseline.")

if __name__ == "__main__":
    run_experiment()

Installing dependencies... (This takes about 30 seconds)

--- System is utilizing: CUDA ---

 PROCESSING DATASET: SCIDOCS


  0%|          | 0/25657 [00:00<?, ?it/s]

  -> Loading SPLADE...


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

  -> Encoding SPLADE vectors...


...

  -> Computing MaxSim...
  -> Loading Contriever Base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  -> Training Interpretable Semantic Resonance (ISR)...


 Table 1: Retrieval, Controllability, and Hardware Footprint Across BEIR Subsets
Dataset    | Model                   | NDCG@10   | DTS (%)   | MCI     | p-value    | Zero-Vals (%)
-----------------------------------------------------------------------------------------------------------------------
Scidocs    | Contriever (Dense)      | 0.5615    | 1.0       | N/A     | -          | 0.0% (Dense)
           | ColBERTv2.0 (Dense)     | 0.6105    | 5.0       | N/A     | 0.168  (ns) | 0.0% (Dense)
           | SPLADE (Sparse)         | 0.5979    | 14.5      | N/A     | 0.325  (ns) | 99.4%
           | ISR-Contriever (Ours)   | 0.5610    | 28.0      | 8.5     | 0.958  (ns) | 96.5%
-----------------------------------------------------------------------------------------------------------------------
Arguana    | Contriever (Dense)      | 0.4628    | 15.0      | N/A     | -          | 0.0% (Dense)
           | ColBERTv2.0 (Dense)     

In [ ]:
# ==========================================
# Ablation Study: Isolating ISR Components
# ==========================================
import os
import gc
import warnings
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F_func
from sentence_transformers import SentenceTransformer
from beir import util
from beir.datasets.data_loader import GenericDataLoader

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--- Running Ablations on: {DEVICE.type.upper()} ---")

# --- Architecture ---
class ISR_Bottleneck(nn.Module):
    def __init__(self, dense_dim=768, sparse_dim=4096):
        super(ISR_Bottleneck, self).__init__()
        self.encoder = nn.Linear(dense_dim, sparse_dim)
        self.relu = nn.ReLU()
        self.decoder = nn.Linear(sparse_dim, dense_dim)

    def forward(self, dense_embeds):
        F = self.relu(self.encoder(dense_embeds))
        return F, self.decoder(F)

def train_isr(model, train_embeddings, epochs=150, l1_lambda=1.5e-6, ip_lambda=3.0):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    mse_loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        F, rec_E = model(train_embeddings)

        # 1. Reconstruction Loss
        rec_loss = mse_loss_fn(rec_E, train_embeddings)
        # 2. Sparsity Loss
        sparse_loss = l1_lambda * torch.norm(F, p=1)

        # 3. Contrastive/IP Loss
        if ip_lambda > 0:
            F_norm = F_func.normalize(F, p=2, dim=1, eps=1e-8)
            sim_F = torch.matmul(F_norm, F_norm.T)
            sim_E = torch.matmul(train_embeddings, train_embeddings.T)
            ip_loss = mse_loss_fn(sim_F, sim_E)
        else:
            ip_loss = 0.0

        total_loss = rec_loss + sparse_loss + (ip_lambda * ip_loss)
        total_loss.backward()
        optimizer.step()
    return model

def compute_ndcg_at_k(retrieved_idx, relevant_idx, k=10):
    dcg = sum(1.0 / math.log2(i + 2) for i, idx in enumerate(retrieved_idx[:k]) if idx in relevant_idx)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant_idx), k)))
    return dcg / idcg if idcg > 0 else 0.0

# --- Execution ---
def run_ablations():
    # Setup Data (SciDocs Only for speed)
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scidocs.zip"
    out_dir = util.download_and_unzip(url, "datasets")
    corpus, queries, qrels = GenericDataLoader(data_folder=out_dir).load(split="test")

    query_ids = list(queries.keys())[:20]
    relevant_doc_ids = []
    for qid in query_ids:
        if qid in qrels: relevant_doc_ids.extend([did for did, score in qrels[qid].items() if score > 0])

    all_doc_ids = list(set(relevant_doc_ids + list(corpus.keys())[:500]))
    doc_id_to_idx = {did: i for i, did in enumerate(all_doc_ids)}

    sample_docs = [corpus[did].get("text") for did in all_doc_ids]
    sample_queries = [queries[qid] for qid in query_ids]

    # Embed Base Vectors
    print("Encoding Base Vectors...")
    base_model = SentenceTransformer('facebook/contriever', device=DEVICE)
    d_dense = F_func.normalize(torch.tensor(base_model.encode(sample_docs, batch_size=64)).to(DEVICE), p=2, dim=1)
    q_dense = F_func.normalize(torch.tensor(base_model.encode(sample_queries, batch_size=64)).to(DEVICE), p=2, dim=1)
    train_embeddings = torch.cat([d_dense, q_dense], dim=0)

    # --- Ablation Configurations ---
    configs = [
        {"name": "Optimal ISR", "k": 4096, "l1": 1.5e-6, "ip": 3.0},
        {"name": "Expanded Dim", "k": 8192, "l1": 1.5e-6, "ip": 3.0},
        {"name": "No IP/Contrastive", "k": 4096, "l1": 1.5e-6, "ip": 0.0},
        {"name": "No Sparsity (L1=0)", "k": 4096, "l1": 0.0, "ip": 3.0}
    ]

    results = []

    for cfg in configs:
        print(f"Training: {cfg['name']} (k={cfg['k']}, L1={cfg['l1']}, IP={cfg['ip']})...")
        model = ISR_Bottleneck(dense_dim=768, sparse_dim=cfg['k']).to(DEVICE)
        model = train_isr(model, train_embeddings, epochs=150, l1_lambda=cfg['l1'], ip_lambda=cfg['ip'])
        model.eval()

        with torch.no_grad():
            q_F, _ = model(q_dense)
            d_F, _ = model(d_dense)

        q_F_norm = F_func.normalize(q_F, p=2, dim=1, eps=1e-8)
        d_F_norm = F_func.normalize(d_F, p=2, dim=1, eps=1e-8)

        scores = torch.matmul(q_F_norm, d_F_norm.T)
        ndcg_list, dts_list = [], []

        for i, qid in enumerate(query_ids):
            if qid not in qrels: continue
            relevant_idx = [doc_id_to_idx[did] for did in qrels[qid].keys() if did in doc_id_to_idx]
            if not relevant_idx: continue

            top_k_idx = torch.topk(scores[i], k=10).indices.tolist()
            ndcg_list.append(compute_ndcg_at_k(top_k_idx, relevant_idx, k=10))

            q_facet = torch.argmax(q_F_norm[i]).item()
            dts_matches = sum(1 for idx in top_k_idx if q_facet in torch.topk(d_F_norm[idx], k=3).indices.tolist())
            dts_list.append((dts_matches / 10.0) * 100)

        sparsity = (1.0 - ((d_F_norm > 0).float().sum(dim=1).mean().item() / cfg['k'])) * 100

        results.append({
            "name": cfg['name'],
            "ndcg": np.mean(ndcg_list),
            "dts": np.mean(dts_list),
            "sparsity": sparsity
        })

    # --- Output Table ---
    print("\n\n=========================================================================")
    print(" Table 2: Ablation Study on ISR Components (SciDocs)")
    print("=========================================================================")
    print(f"{'Configuration':<25} | {'NDCG@10':<9} | {'DTS (%)':<9} | {'Zero-Vals (%)'}")
    print("-" * 65)
    for r in results:
        print(f"{r['name']:<25} | {r['ndcg']:<9.4f} | {r['dts']:<9.1f} | {r['sparsity']:.1f}%")
    print("=========================================================================")

if __name__ == "__main__":
    run_ablations()

--- Running Ablations on: CUDA ---


  0%|          | 0/25657 [00:00<?, ?it/s]

Encoding Base Vectors...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Training: Optimal ISR (k=4096, L1=1.5e-06, IP=3.0)...
Training: Expanded Dim (k=8192, L1=1.5e-06, IP=3.0)...
Training: No IP/Contrastive (k=4096, L1=1.5e-06, IP=0.0)...
Training: No Sparsity (L1=0) (k=4096, L1=0.0, IP=3.0)...


 Table 2: Ablation Study on ISR Components (SciDocs)
Configuration             | NDCG@10   | DTS (%)   | Zero-Vals (%)
-----------------------------------------------------------------
Optimal ISR               | 0.4177    | 85.5      | 97.4%
Expanded Dim              | 0.3968    | 89.5      | 98.8%
No IP/Contrastive         | 0.0500    | 100.0     | 100.0%
No Sparsity (L1=0)        | 0.3946    | 6.0       | 82.0%
